# ROI H5 Dataset Visualizer
Interactive visualization of paired raw/mask patches from ROI segmentation training data

In [4]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from ipywidgets import interact, IntSlider, Dropdown, VBox, HBox, Output, Label
import ipywidgets as widgets
from IPython.display import display, clear_output

In [ ]:
# Set the path to your ROI H5 file
H5_PATH = "/mnt/jean-zay/data_segment_roi/roi_segmentation.h5"

In [6]:
# Open H5 file and inspect structure
h5f = h5py.File(H5_PATH, 'r')

print("H5 File Structure:")
print(f"Root keys: {list(h5f.keys())}")
for split in h5f.keys():
    print(f"\n{split}/:")
    for key in h5f[split].keys():
        ds = h5f[split][key]
        print(f"  {key}: {ds.shape}, dtype={ds.dtype}, "
              f"min={ds[0].min():.4f}, max={ds[0].max():.4f}")

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '/mnt/jean-zay/roi_training/roi_segmentation.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:
def random_label_cmap(n=2048):
    """Generate a random colormap for integer label masks with 0 mapped to black."""
    rng = np.random.default_rng(42)
    colors = rng.uniform(0.2, 1.0, size=(n, 3))
    colors[0] = [0, 0, 0]  # background = black
    return ListedColormap(colors)

LABEL_CMAP = random_label_cmap()


class RoiH5Visualizer:
    def __init__(self, h5_file):
        self.h5f = h5_file
        self.current_split = 'train'

        self.raw = self.h5f[self.current_split]['raw']
        self.mask = self.h5f[self.current_split]['mask']
        self.n_samples = self.raw.shape[0]

        self.output = Output()
        self.stats_output = Output()

    def switch_split(self, split):
        self.current_split = split
        self.raw = self.h5f[split]['raw']
        self.mask = self.h5f[split]['mask']
        self.n_samples = self.raw.shape[0]

    def visualize(self, sample_idx, show_overlay):
        with self.output:
            clear_output(wait=True)

            raw_patch = self.raw[sample_idx]
            mask_patch = self.mask[sample_idx]

            ncols = 3 if show_overlay else 2
            fig, axes = plt.subplots(1, ncols, figsize=(6 * ncols, 6))

            axes[0].imshow(raw_patch, cmap='gray')
            axes[0].set_title(f'Raw — Sample {sample_idx}')
            axes[0].axis('off')

            n_labels = len(np.unique(mask_patch))
            axes[1].imshow(mask_patch, cmap=LABEL_CMAP, interpolation='nearest')
            axes[1].set_title(f'Mask — {n_labels} labels')
            axes[1].axis('off')

            if show_overlay:
                axes[2].imshow(raw_patch, cmap='gray')
                mask_rgb = LABEL_CMAP(mask_patch.astype(int) % LABEL_CMAP.N)
                mask_alpha = np.where(mask_patch > 0, 0.4, 0.0)
                mask_rgba = np.concatenate(
                    [mask_rgb[..., :3], mask_alpha[..., None]], axis=-1
                )
                axes[2].imshow(mask_rgba)
                axes[2].set_title('Overlay')
                axes[2].axis('off')

            plt.tight_layout()
            plt.show()

        with self.stats_output:
            clear_output(wait=True)
            print("=" * 60)
            print(f"Patch {sample_idx} — {self.current_split} split")
            print("=" * 60)
            print(f"Raw  shape: {raw_patch.shape}, dtype: {raw_patch.dtype}")
            print(f"Raw  — min: {raw_patch.min():.4f}, max: {raw_patch.max():.4f}, "
                  f"mean: {raw_patch.mean():.4f}, std: {raw_patch.std():.4f}")
            print(f"Mask shape: {mask_patch.shape}, dtype: {mask_patch.dtype}")
            labels = np.unique(mask_patch)
            print(f"Mask — {len(labels)} unique labels, "
                  f"min: {labels.min()}, max: {labels.max()}")
            fg_frac = np.count_nonzero(mask_patch) / mask_patch.size * 100
            print(f"Foreground: {fg_frac:.1f}%")
            print("=" * 60)

    def create_widgets(self):
        split_dropdown = Dropdown(
            options=list(self.h5f.keys()),
            value=self.current_split,
            description='Split:'
        )

        sample_slider = IntSlider(
            min=0, max=self.n_samples - 1, step=1, value=0,
            description='Sample:',
            continuous_update=False
        )

        overlay_checkbox = widgets.Checkbox(
            value=True,
            description='Show Overlay'
        )

        def on_split_change(change):
            self.switch_split(change['new'])
            sample_slider.max = self.n_samples - 1
            sample_slider.value = min(sample_slider.value, self.n_samples - 1)
            self.visualize(sample_slider.value, overlay_checkbox.value)

        split_dropdown.observe(on_split_change, names='value')

        def update(sample_idx, show_overlay):
            self.visualize(sample_idx, show_overlay)

        widgets.interactive(
            update,
            sample_idx=sample_slider,
            show_overlay=overlay_checkbox,
        )

        controls = VBox([
            Label(f'Dataset: {self.n_samples} patches, shape: {self.raw.shape}'),
            split_dropdown,
            sample_slider,
            overlay_checkbox,
        ])

        display(controls)
        display(self.output)
        display(self.stats_output)

        self.visualize(0, True)

In [ ]:
# Create and display the visualizer
viz = RoiH5Visualizer(h5f)
viz.create_widgets()

Output()

Output()